# 🔍 SQL Murder Mystery
**Ciudad:** SQL City &nbsp;|&nbsp; **Fecha del crimen:** 15 enero 2018 &nbsp;|&nbsp; **Tipo:** Homicidio

---

## Fase 1 — Escena del crimen

Lo primero es buscar el informe del crimen en la base de datos.

```sql
SELECT *
FROM crime_scene_report
WHERE city = 'SQL City'
  AND type = 'murder'
  AND date = '20180115';
```

> 📋 **Resultado:** Las cámaras de seguridad registraron **dos testigos**:
> - El primero vive en la **última casa de Northwestern Dr.**
> - La segunda, **Annabel**, vive en alguna parte de **Franklin Ave.**

---
## Fase 2 — Localización de testigos

Busco a los dos testigos por separado.

**Testigo 1 — Annabel (Franklin Ave)**
```sql
SELECT *
FROM person
WHERE address_street_name = 'Franklin Ave'
  AND name LIKE '%Annabel%';
```

| ID | Nombre | License ID | Dirección |
|---|---|---|---|
| 16371 | Annabel Miller | 490173 | 103 Franklin Ave |

**Testigo 2 — Último residente de Northwestern Dr.**
```sql
SELECT *
FROM person
WHERE address_street_name = 'Northwestern Dr'
ORDER BY address_number DESC
LIMIT 5;
```

| ID | Nombre | License ID | Número |
|---|---|---|---|
| 14887 | Morty Schapiro | 118009 | 4919 |

---
## Fase 3 — Declaraciones de los testigos

Consulto las entrevistas de ambos.

```sql
SELECT *
FROM interview
WHERE person_id = 16371
   OR person_id = 14887;
```

> 🗣️ **Morty Schapiro (14887):**
> *"Escuché un disparo y vi a un hombre correr. Llevaba una bolsa del **Get Fit Now Gym**. El número de membresía empezaba por **«48Z»** — solo los miembros dorados tienen esas bolsas. El coche tenía una matrícula con **«H42W»**."*

> 🗣️ **Annabel Miller (16371):**
> *"Vi el asesinato. Reconocí al asesino: era del **gimnasio** donde yo entrenaba. Lo vi el **9 de enero**."*

---
## Fase 4 — Identificación del asesino

Con las pistas del testigo, busco en los registros del gimnasio y en matrículas.

**Registros del gimnasio el 9 de enero — membresía que empieza por 48Z**
```sql
SELECT *
FROM get_fit_now_check_in
WHERE membership_id LIKE '48Z%'
  AND check_in_date = '20180109';
```

| ID membresía | Nombre | Estado | Entrada | Salida |
|---|---|---|---|---|
| 48Z7A | Joe Germuska | Gold | 16:00 | 17:30 |
| 48Z55 | Jeremy Bowers | Gold | 15:30 | 17:00 |

**Busco la matrícula con «H42W» entre hombres**
```sql
SELECT *
FROM drivers_license
WHERE gender = 'male'
  AND plate_number LIKE '%H42W%';
```

| License ID | Edad | Pelo | Matrícula | Vehículo |
|---|---|---|---|---|
| 423327 | 30 | Castaño | 0H42W2 | Chevrolet Spark LS |
| 664760 | 21 | Negro | 4H42WR | Nissan Altima |

**Cruzo ambos resultados — ¿quién aparece en los dos?**
```sql
SELECT *
FROM get_fit_now_member
WHERE person_id IN ('51739', '67318');
```

| ID membresía | Nombre | Estado |
|---|---|---|
| 48Z55 | **Jeremy Bowers** | Gold ✓ |

```sql
INSERT INTO solution VALUES (1, 'Jeremy Bowers');
SELECT value FROM solution;
```

✅ **¡Correcto! Jeremy Bowers es el asesino.** Pero el sistema avisa de que hay más...

---
## Fase 5 — La mente criminal

Consulto la entrevista del propio asesino para encontrar al verdadero cerebro.

```sql
SELECT *
FROM interview
WHERE person_id = 67318;
```

> 🗣️ **Jeremy Bowers (67318):**
> *"Me contrató una mujer con mucho dinero. No sé su nombre, pero mide entre **5'5" y 5'7"**, tiene el **pelo rojo** y conduce un **Tesla Model S**. Asistió al concierto de la **SQL Symphony Orchestra 3 veces en diciembre de 2017**."*

**Busco su perfil físico y vehículo**
```sql
SELECT *
FROM drivers_license
WHERE gender = 'female'
  AND hair_color = 'red'
  AND height BETWEEN 65 AND 67
  AND car_make = 'Tesla'
  AND car_model = 'Model S';
```

| License ID | Nombre (cruzado) | Altura | Ojos | Matrícula |
|---|---|---|---|---|
| 918773 | Red Korb | 65" | Negro | 917UU3 |
| 291182 | Regina George | 66" | Azul | 08CM64 |
| 202298 | Miranda Priestly | 66" | Verde | 500123 |

**¿Cuál de las tres fue al concierto 3 veces en diciembre?**
```sql
SELECT person_id, event_name, COUNT(*) AS veces
FROM facebook_event_checkin
WHERE person_id IN ('78881', '90700', '99716')
GROUP BY person_id, event_name;
```

| ID | Evento | Veces |
|---|---|---|
| 99716 | SQL Symphony Concert | **3** ✓ |

```sql
INSERT INTO solution VALUES (1, 'Miranda Priestly');
SELECT value FROM solution;
```

---

### 🏆 ¡Caso resuelto!

**Asesino contratado:** Jeremy Bowers  
**Cerebro del crimen:** Miranda Priestly

> *"¡Enhorabuena! Todo el mundo en SQL City te aclama como el mejor detective de SQL de todos los tiempos."*